## Protein Structure Prediction using LambdaFold model

In this notebook, we demonstrate how to predict protein 3D structure using LambdaFold model

In [1]:
%cd ..

/net/kihara/scratch/nibtehaz/Prot-LAMBDA


In [2]:
## load model

from ProtLAMBDA import DistFormer, ProtLAMBDA, LambdaFold, load_fastas
import torch

plm = ProtLAMBDA()
plm.load_state_dict(torch.load('/net/kihara/scratch/nibtehaz/Prot-LAMBDA/params/ProtLAMBDA.pth',map_location='cpu'))
dst = DistFormer()
dst.load_state_dict(torch.load('/net/kihara/scratch/nibtehaz/Prot-LAMBDA/params/DistFormer.pth',map_location='cpu'))
lmbd = LambdaFold(plm, dst)
lmbd.load_state_dict(torch.load('/net/kihara/scratch/nibtehaz/Prot-LAMBDA/params/LambdaFold.pth',map_location='cpu'),strict=False)

dvc = 'cuda:3'

plm.to(dvc)
dst.to(dvc)
lmbd.to(dvc)
plm.eval()
dst.eval();
lmbd.eval();

/tmp/ipykernel_1802133/3101010131.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  plm.load_state_dict(torch.load('/net/kihara/scratch/nibtehaz/Prot-LAMBDA/params/ProtLAM

In [3]:
## Load seqs 

from ProtLAMBDA import load_fastas

seqs = load_fastas('sample_protein')

seqs

[['7fjs_L',
  'RTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGECQIVLTQSPSSLAVSVGEKVTLSCKSSQSLLYSNNQKNYLAWYQQKSGRSPKLLLHWTSTRESGVPDRFTGSGSGTDFTLTISSVKAEDLAVYYCQQYYTYPWTFGGGTKLEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC'],
 ['7x8v_A',
  'GPGHMMAAEAWRSRFRERVVEAAERWESVGESLATALTHLKSPMHAGDEEEAAAARTRIQLAMGELVDASRNLASAMSLMKVAELLALHGGSVNPSTHLGEISLLGDQYLAERNAGIKLLEAGKDARKAYISVDGCRGNLDAILLLLDHPRVPCVDDFIEEELFVAGDNLQGAIGNAKLGTERAVGARQDVSGAN'],
 ['T1089',
  'MDDNVNDFRPVPYPDEVVGLNPDPDFEPIWEIASPTITVFSSKASNDISYRVPAIAVTKKGSILVFCEARYGTWQDKAGRTDILMKRSTDKGITWTEKNLTNQATSSKLSYMDPTVVVDQVTGKIFLFTSLWDAVGKESAKQGYNNRAIMYTSEDDGLNWTRKDLTDEVEIGIFSGATRMIGSFGPGSGVQMTSSEQYKNRLIVPIRTFKVNEAAGTVSNGGNTAMWSDDNGGTWETGQPNKSGEWMVTEAPDGALIGNIRYNGHRQNYVSTDGGAKWPSFSDYDPIALPTPAKGCAGSVIVKDGWMYYCGAKGIIETTAHDDRGILYLAKAKFFGGHSHTFDPADHMVLYDKAAGYTCMALLPDGDMAIVAELGNEPGFQKLSTRPAEWMRLELFILSTKKPL'],
 ['T1034',
  'LILN

In [4]:
## Predict all the intermidiate structures

with torch.no_grad():
    dists, strucs, plddt, pdb = lmbd(seqs[3][1], return_all=True, chunk_size=32)

In [5]:
# writing the pdb files to disk

for i in range(len(pdb)):
    with open(f'{seqs[3][0]}_{i}.pdb', 'w') as f:
        f.write(pdb[i])

In [6]:
## Alternatively, just predict the final structure

with torch.no_grad():
    dists, strucs, plddt, pdb = lmbd(seqs[3][1], return_all=False, chunk_size=32)

In [7]:
print(pdb[0])

MODEL     1
ATOM      1  N   LEU A   1     -11.624  21.423   2.792  1.00 50.00           N  
ATOM      2  CA  LEU A   1     -10.857  20.424   2.057  1.00 50.00           C  
ATOM      3  C   LEU A   1     -11.074  19.033   2.643  1.00 50.00           C  
ATOM      4  CB  LEU A   1      -9.366  20.771   2.074  1.00 50.00           C  
ATOM      5  O   LEU A   1     -10.782  18.797   3.818  1.00 50.00           O  
ATOM      6  CG  LEU A   1      -8.672  20.847   0.714  1.00 50.00           C  
ATOM      7  CD1 LEU A   1      -8.139  22.255   0.468  1.00 50.00           C  
ATOM      8  CD2 LEU A   1      -7.546  19.822   0.630  1.00 50.00           C  
ATOM      9  N   ILE A   2     -12.241  18.692   3.002  1.00 50.00           N  
ATOM     10  CA  ILE A   2     -12.666  17.325   3.282  1.00 50.00           C  
ATOM     11  C   ILE A   2     -11.744  16.341   2.565  1.00 50.00           C  
ATOM     12  CB  ILE A   2     -14.133  17.092   2.857  1.00 50.00           C  
ATOM     13  O  